In [1]:
"""
Train probes
"""
None

In [2]:
"""
Imports
"""
import torch
from datasets import load_dataset
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import gc
import pickle
import importlib
import cupy
import cuml
import sklearn
import yaml

from utils.memory import check_memory, clear_all_cuda_memory
from local_runtime import load_model_and_tokenizer, load_custom_forward_pass
from utils.probes import run_projections, check_max_seq_len
from local_runtime import run_and_export_states, OUT, atomic_json
from pathlib import Path
import hashlib, json

main_device = 'cuda:0'
seed = 123

clear_all_cuda_memory()
check_memory()

ws = '/root/prompt-injection-as-role-confusion'

All CUDA memory cleared on all devices.
Device 0: NVIDIA A100-SXM4-80GB
  Allocated: 0.00 GB
  Reserved: 0.00 GB
  Total: 79.25 GB



# Load model

In [3]:
"""
Load the base tokenizer/model
"""
model_prefix = 'gptoss-20b'
tokenizer, model, model_architecture, model_n_layers = load_model_and_tokenizer(model_prefix, device = main_device)

check_memory()

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Loaded local 20B: BF16 dequantized/eager experts BF16 activations, eager attention


Device 0: NVIDIA A100-SXM4-80GB
  Allocated: 38.96 GB
  Reserved: 39.55 GB
  Total: 79.25 GB



In [4]:
"""
Load custom forward pass and verify equality to base model forward pass
"""
run_forward_with_hs = load_custom_forward_pass(model_architecture, model, tokenizer)

Actual-model forward hook validation: {'exact_logits': True, 'max_absolute_logit_difference': 0.0, 'layers_captured': 24, 'capture_point': 'post_attention_layernorm output'}


In [5]:
"""
Test generation is sensible
"""
def test_generation():
    conv = tokenizer.apply_chat_template(
        [{"role": "user", "content": "Write a haiku about GPUs"},],
        tokenize = False,
        enable_thinking = True,
        add_generation_prompt = True
    )
    inputs = tokenizer(conv, return_tensors = 'pt')
    gen_ids = model.generate(inputs['input_ids'].to(main_device), max_new_tokens = 100, do_sample = False)
    print(tokenizer.batch_decode(gen_ids, skip_special_tokens = False)[0])    

test_generation()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-10-19

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Write a haiku about GPUs<|end|><|start|>assistant<|channel|>analysis<|message|>The user wants a haiku about GPUs. Haiku is 5-7-5 syllable structure. Should be about GPUs. Let's craft a haiku. 5 syllables first line, 7 second, 5 third. Let's think: "Pixels dance, bright" maybe. Count syllables: "Pixel" 2, "dance" 1, "bright" 1 => 4. Need 5. Maybe "Pixels dance, bright glow" Count


# Train probes

## Prep dataset

In [6]:
"""
Load SFT dataset
"""
n_sample_size = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['n_sample_size']

def load_raw_ds():

    def get_c4():
        return load_dataset('allenai/c4', 'en', split = 'validation', streaming = True).shuffle(seed = seed, buffer_size = 50_000)
    
    def get_dolma3():
        return load_dataset('allenai/dolma3_mix-150B-1025', split = 'train', revision = '3a8349c', streaming = True).shuffle(seed = seed, buffer_size = 50_000)
    
    def get_data(ds, n_samples, data_source):
        raw_data = []
        ds_iter = iter(ds)
        for _ in range(n_samples):
            sample = next(ds_iter, None)
            if sample is None:
                break
            raw_data.append({'text': sample['text'], 'source': data_source})
        return raw_data
    
    return get_data(get_c4(), int(n_sample_size * .25), 'c4')  + get_data(get_dolma3(), int(n_sample_size * .75), 'dolma3')

raw_data = json.loads((OUT/'raw_training_data.json').read_text())
assert len(raw_data)==249
print('Training source counts:',pd.DataFrame(raw_data).source.value_counts().to_dict())

Training source counts: {'dolma3': 187, 'c4': 62}


In [7]:
"""
Test rendering
"""
import utils.role_templates
importlib.reload(utils.role_templates)
from utils.role_templates import render_single_message, render_mixed_cot

def test_render():
    print(tokenizer.apply_chat_template(
        [
            {'role': 'user', 'content': 'Hi! I am a dog and I like to bark'},
            {'role': 'assistant', 'content': 'Hello! What a lovely dog you are!'}
        ],
        tokenize = False, padding = 'max_length', truncation = True, max_length = 512, add_generation_prompt = True,
        enable_thinking = True
    ))
    print('--')
    print(render_single_message(model_prefix, role = 'user', content ='Hi'))
    print('--')
    print(render_mixed_cot(model_prefix, 'The user...', 'Yes!'))
    return True

test_render()

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-10-19

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi! I am a dog and I like to bark<|end|><|start|>assistant<|channel|>final<|message|>Hello! What a lovely dog you are!<|end|><|start|>assistant
--
<|start|>user<|message|>Hi<|end|>
--
<|start|>assistant<|channel|>analysis<|message|>The user...<|end|><|start|>assistant<|channel|>final<|message|>Yes!<|end|>


True

In [8]:
"""
Validate forward passes work
"""
# Define opening texts to append at the start of each role for training. If multiple, will randomize equally.
train_prefixes = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['train_prefixes']
for p in train_prefixes:
    print(p)

@torch.no_grad()
def test1():
    conv = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'Hi stinky'}],
        tokenize = False,
        add_generation_prompt = True
    )
    inputs = tokenizer(conv, add_special_tokens = True, return_tensors = 'pt')
    gen_ids = model.generate(inputs['input_ids'].to(main_device), attention_mask = inputs['attention_mask'].to(main_device), max_new_tokens = 500, do_sample = False)
    df = pd.DataFrame({'token_id': gen_ids[0].tolist(), 'token': tokenizer.convert_ids_to_tokens(gen_ids[0].tolist()),})
    print(tokenizer.decode(df['token_id'].tolist(), skip_special_tokens = False))
    return df

@torch.no_grad()
def test2():
    """Quick test - can the model run forward passes correctly?"""
    for tr_prefix in train_prefixes:
        conv = tr_prefix + render_single_message(model_prefix, 'user', 'Where is Atlanta?') +\
            (
            '<|start|>assistant<|channel|>analysis<|message|>' # gpt-oss-*
            # '<|im_start|>assistant\n<think></think>' # Nemotron
            # '<|assistant|>\n<think>' # GLM-4.6V-Flash
            # '<|im_start|>assistant\n<think>\n' # Qwen3
            # '\n<|begin_assistant|>\nHere are my reasoning steps:\n' # Apriel-1.6-15B-Thinker
            # '<|assistant|><think>' # GLM-4.7-Flash
            )
        inputs = tokenizer(conv, add_special_tokens = False, return_tensors = 'pt')
        gen_ids = model.generate(inputs['input_ids'].to(main_device), attention_mask = inputs['attention_mask'].to(main_device), max_new_tokens = 500, do_sample = False)
        print(tokenizer.decode(gen_ids[0], skip_special_tokens = False))
        print('\n')

test1()
test2()

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-10-19

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi stinky<|end|><|start|>assistant<|channel|>analysis<|message|>The user says "Hi stinky". They are addressing the assistant as "stinky". The user might be joking or insulting. The assistant should respond politely, maybe with a friendly greeting. The user might want a friendly response. The assistant should not respond with negativity. The user might be testing the assistant's response to a nickname. The assistant should respond politely, maybe ask how they can help. The user might want a conversation. The assistant should respond in a friendly tone. The user might be using a nickname. The assistant should respond politely. The user might want to talk about something. The assistant should respond with a friendly 

<|start|>user<|message|>Where is Atlanta?<|end|><|start|>assistant<|channel|>analysis<|message|>We need to answer: "Where is Atlanta?" The user asks location. We should respond with location: Atlanta is a city in the state of Georgia, USA. It's in the southeastern United States. Provide details.<|end|><|start|>assistant<|channel|>final<|message|>**Atlanta** is a major city located in the southeastern United States.  
- **State:** Georgia  
- **Country:** United States of America  
- **Geographic position:** Roughly in the central part of Georgia, about 70 mi (110 km) south of the state capital, Atlanta sits on the eastern side of the Appalachian foothills, near the intersection of several major highways and rail lines.  

It’s the state’s largest city and a key economic, cultural, and transportation hub for the region.<|return|>




In [9]:
"""
Create sample sequences
"""
SEQLEN = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['seq_len']
NESTED_REASONING = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['nested_reasoning']
GENERALIZE_PREFIX = False

def get_sample_seqs_for_input_seq(probe_text, partner_text, prefix = ''):
    """
    Helper to convert a single input sequence into all examples

    Params
        @probe_text: The text we're extracting states from (appears in all roles)
        @partner_text: Random paired text (only used in merged sample's assistant position)
    """
    seqs = []

    gen_prefix = partner_text if GENERALIZE_PREFIX else ''

    # Only test system roles for gptoss-20b for time
    if model_prefix in ['gptoss-20b']:
        seqs.append({
            'role': 'system',
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = 'system', content = probe_text)
        })

    for role in ['user', 'tool', 'cot']:
        seqs.append({
            'role': role,
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = role, content = probe_text)
        })

    # If merge AND LRM, render with CoT prefix
    if NESTED_REASONING:
        seqs.append({
            'role': 'assistant',
            'prompt': prefix + render_mixed_cot(model_prefix, cot = partner_text, assistant = probe_text)
        })
    # Else render standalone
    else:
        seqs.append({
            'role': 'assistant',
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = 'assistant', content = probe_text)
        })

    return seqs

def build_sample_seqs(train_prefixes):
    """
    Build all sample sequences
    """
    truncated_texts = tokenizer.batch_decode(tokenizer([t['text'] for t in raw_data], add_special_tokens = False, padding = False, truncation = True, max_length = SEQLEN).input_ids)
    n_seqs = len(truncated_texts)

    np.random.seed(seed)
    partner_lengths = ((np.random.beta(0.5, 4.0, size = n_seqs) * (SEQLEN/2 + 1)).astype(int)).tolist()
    partner_texts = [
        tokenizer.decode(tokenizer(text['text'], add_special_tokens = False, padding = False, truncation = True, max_length = int(partner_lengths[i])).input_ids)
        for i, text in enumerate(raw_data)
    ]
    perm = np.random.permutation(n_seqs)
    while np.any(perm == np.arange(n_seqs)):
        perm = np.random.permutation(n_seqs)

    sampled_prefixes = np.random.choice(train_prefixes, size = n_seqs)

    input_list = []
    for base_ix, base_text in enumerate(truncated_texts):
        partner_text = partner_texts[int(perm[base_ix])].strip()
        prefix = sampled_prefixes[base_ix]

        for seq in get_sample_seqs_for_input_seq(base_text, partner_text, prefix):
            row = {'question_ix': base_ix, 'question': base_text, **seq}
            input_list.append(row)

    input_df = pd.DataFrame(input_list).assign(prompt_ix = lambda df: list(range(len(df))))
    return input_df

input_df = build_sample_seqs(train_prefixes = train_prefixes)
display(input_df)

for p in [row['prompt'] for row in input_df.pipe(lambda df: df[df['question_ix'] == 2]).to_dict('records')]:
    print(p)
    print("=" * 80)
input_df.to_parquet(OUT/'training_inputs.parquet',index=False)


,question_ix,question,role,prompt,prompt_ix
0,0,The Irish Government will deliver a multi-annu...,system,<|start|>system<|message|>The Irish Government...,0
1,0,The Irish Government will deliver a multi-annu...,user,<|start|>user<|message|>The Irish Government w...,1
2,0,The Irish Government will deliver a multi-annu...,tool,<|start|>functions. to=assistant<|channel|>com...,2
3,0,The Irish Government will deliver a multi-annu...,cot,<|start|>assistant<|channel|>analysis<|message...,3
4,0,The Irish Government will deliver a multi-annu...,assistant,<|start|>assistant<|channel|>final<|message|>T...,4
...,...,...,...,...,...
1240,248,C. W. Stanford Auditorium Addition\n\nOrange C...,system,<|start|>system<|message|>C. W. Stanford Audit...,1240
1241,248,C. W. Stanford Auditorium Addition\n\nOrange C...,user,<|start|>user<|message|>C. W. Stanford Auditor...,1241
1242,248,C. W. Stanford Auditorium Addition\n\nOrange C...,tool,<|start|>functions. to=assistant<|channel|>com...,1242
1243,248,C. W. Stanford Auditorium Addition\n\nOrange C...,cot,<|start|>assistant<|channel|>analysis<|message...,1243


<|start|>system<|message|>In mythology, the Muses were nine goddesses who symbolized the arts and sciences. Today, a muse is a person who serves as an artist's inspiration. We are inspired to build precision cutting machines because our hearts and souls belong to the creative application of scientific principles to design and develop machines and manufacturing processes to the benefit of our customers. Our vocation and global vision enable us to be a dynamic and outstanding world company.
Creating metal artwork has a life long impact on society and building Modern Marvels in metal can take on many forms from signs to sculpture. So with that in mind we ask you to become our muse by cutting plate on Messer machines to make your vision a reality.<|end|>
<|start|>user<|message|>In mythology, the Muses were nine goddesses who symbolized the arts and sciences. Today, a muse is a person who serves as an artist's inspiration. We are inspired to build precision cutting machines because our hear

In [10]:
""" 
Load dataset into a dataloader which returns original tokens - important for BPE tokenizers to reconstruct the correct string later
"""
from utils.dataset import ReconstructableTextDataset, stack_collate
from torch.utils.data import DataLoader

max_seqlen = check_max_seq_len(tokenizer, input_df['prompt'].tolist())
train_dl = DataLoader(
    ReconstructableTextDataset(input_df['prompt'].tolist(), tokenizer, max_length = max_seqlen, prompt_ix = input_df['prompt_ix'].tolist()),
    batch_size = 4,
    shuffle = False,
    collate_fn = stack_collate
)

## Get hidden states

In [11]:
layers_to_probe = list(range(0,model_n_layers,2))
training_signature = hashlib.sha256(json.dumps({
    'raw_data':raw_data,'config':yaml.safe_load(open('config/probe.yaml'))[model_prefix],
    'backend':'BF16-dequantized/eager','layers':layers_to_probe,
},sort_keys=True).encode()).hexdigest()[:20]
probe_checkpoints = OUT/'probe-checkpoints'/training_signature
probe_checkpoints.mkdir(parents=True,exist_ok=True)
training_done = probe_checkpoints/'complete.json'
training_cached = training_done.exists() and len(list(probe_checkpoints.glob('*.pkl')))==96
training_activation_path = None
if training_cached:
    print('All 96 probe checkpoints found; skipping repeated activation extraction.')
    all_probe_hs = {}
else:
    """
    Run forward passes
    """
    layers_to_probe = list(range(0, model_n_layers, 4)) if model_n_layers >= 30 else list(range(0, model_n_layers, 2))
    res = run_and_export_states(model, tokenizer, run_model_return_states = run_forward_with_hs, dl = train_dl, layers_to_keep_acts = layers_to_probe)
    
    # Convert to f16 for cupy compatability
    all_probe_hs = res['all_hs'].to(torch.float16)
    all_probe_hs = {layer_ix: all_probe_hs[:, save_ix, :] for save_ix, layer_ix in enumerate(layers_to_probe)}
    del res['all_hs']
    gc.collect()
    training_activation_path = res['cache_path']


Stream actual-model activations:   0%|          | 0/312 [00:00<?, ?it/s]

Stream actual-model activations:   0%|          | 1/312 [00:01<06:42,  1.29s/it]

Stream actual-model activations:   1%|          | 2/312 [00:02<06:18,  1.22s/it]

Stream actual-model activations:   1%|          | 3/312 [00:03<06:01,  1.17s/it]

Stream actual-model activations:   1%|▏         | 4/312 [00:04<05:55,  1.16s/it]

Stream actual-model activations:   2%|▏         | 5/312 [00:05<05:45,  1.13s/it]

... (305 outputs left out)

Stream actual-model activations: 100%|█████████▉| 311/312 [05:39<00:01,  1.16s/it]

Stream actual-model activations: 100%|██████████| 312/312 [05:39<00:00,  1.04s/it]

Stream actual-model activations: 100%|██████████| 312/312 [05:39<00:00,  1.09s/it]

Completed disk-backed activation map: (12, 726109, 2880) GiB 46.74


## Label roles


In [13]:
if training_cached:
    probe_sample_df = pd.read_parquet(OUT/'training_labels.parquet')
else:
    """
    Take each token and label it with its correct role (user/assistant/system/tool/cot); check the counts are as expected
    """
    import utils.role_assignments
    importlib.reload(utils.role_assignments)
    from utils.role_assignments import label_content_roles
    from utils.substring_assignments import flag_message_types
    
    probe_sample_df = (
        # Flag all roles
        label_content_roles(model_prefix, res['sample_df'])    
        .assign(sample_ix = lambda df: range(0, len(df)))
        # Flag those that match target role (for retain later)
        .merge(input_df[['prompt_ix', 'role']].rename(columns = {'role': 'target_role'}), how = 'inner', on = 'prompt_ix')
        .assign(match_target_role = lambda df: np.where(df['role'] == df['target_role'], True, False))
        # Flag those that match the prepend (for drop later)
        .pipe(lambda df: flag_message_types(df, train_prefixes, allow_ambiguous = True))
        .drop(columns = 'base_message')
        # Drop prepend text + non-content tags
        .pipe(lambda df: df[(df['base_message_ix'].isna()) & (df['is_content'] == True) & (df['role'].notna()) & (df['match_target_role'])])
    )
    
    # Verify role counts are accurate (should be exactly equal for most models, except when models where role tags can be merged with content into single toks)
    display(probe_sample_df.groupby('role', as_index = False).agg(count = ('sample_ix', 'count')))
    
    # Validate roles are flagged correctly
    display(
        probe_sample_df\
        .pipe(lambda df: df[df['prompt_ix'] <= 14])\
        .groupby(['prompt_ix', 'seg_ix', 'role'], as_index = False)\
        .agg(combined_text = ('token', ''.join))\
        .assign(eot = lambda df: df['combined_text'].str[-30:])
    )
    probe_sample_df.to_parquet(OUT/'training_labels.parquet',index=False)

# The model is not used by logistic regression; release its GPU weights temporarily.
model.to('cpu')
clear_all_cuda_memory()


,role,count
0,assistant,143678
1,cot,143678
2,system,143678
3,tool,143678
4,user,143678


,prompt_ix,seg_ix,role,combined_text,eot
0,0,0,system,The Irish Government will deliver a multi-annu...,"obs and development,"" he said."
1,1,0,user,The Irish Government will deliver a multi-annu...,"obs and development,"" he said."
2,2,0,tool,The Irish Government will deliver a multi-annu...,"obs and development,"" he said."
3,3,0,cot,The Irish Government will deliver a multi-annu...,"obs and development,"" he said."
4,4,0,assistant,The Irish Government will deliver a multi-annu...,"obs and development,"" he said."
5,5,0,system,(1) (National Center for Supercomputing Applic...,ty Association) See ICSA Labs.
6,6,0,user,(1) (National Center for Supercomputing Applic...,ty Association) See ICSA Labs.
7,7,0,tool,(1) (National Center for Supercomputing Applic...,ty Association) See ICSA Labs.
8,8,0,cot,(1) (National Center for Supercomputing Applic...,ty Association) See ICSA Labs.
9,9,0,assistant,(1) (National Center for Supercomputing Applic...,ty Association) See ICSA Labs.


All CUDA memory cleared on all devices.


## Run probes

In [14]:
"""
First run a quick grid search over hyperparmeters
"""
SKIP_FIRST_N = 32 if NESTED_REASONING else 0

def fit_lr(x_train, y_train, x_test, y_test, add_scaling = False, **lr_params):
    """
    Fit a probe
    """
    steps = []
    if add_scaling:
        steps.append(('scaler', cuml.preprocessing.StandardScaler()))
    steps.append(('clf', cuml.linear_model.LogisticRegression(penalty = 'l2', max_iter = 5_000, linesearch_max_iter = 100, fit_intercept = True, **lr_params)))
    lr_model = sklearn.pipeline.Pipeline(steps)
    lr_model.fit(x_train, y_train)
    accuracy = lr_model.score(x_test, y_test)
    y_test_pred = lr_model.predict(x_test)
    y_test_prob = lr_model.predict_proba(x_test)
    nll = cuml.metrics.log_loss(y_test, y_test_prob,)
    return lr_model, accuracy, nll, y_test_pred

def get_probe_result(sample_df, layer_hs, roles_map, add_scaling = False, **lr_params):
    """
    Get probe results for a single layer and label combination

    Params:
        @sample_df: The sample-level df; with a column `sample_ix` indicating the token order of 0...T-1;
            the actual df may be shorter due to pre-filters
        @layer_hs: A tensor of probe hidden states for a layer, of T x D
        @roles_map: The mapping order of the roles; a dict {}

    Description:
        Trains only on content space for given roles
    """
    # Train/test split
    prompt_ix_train, prompt_ix_test = cuml.train_test_split(sample_df['prompt_ix'].unique(), test_size = 0.1, random_state = seed)
    train_df = sample_df[sample_df['prompt_ix'].isin(prompt_ix_train)]
    test_df = sample_df[sample_df['prompt_ix'].isin(prompt_ix_test)]

    # Get y labels
    role_labels_train_cp = cupy.asarray([roles_map[r] for r in train_df['role']])
    role_labels_test_cp = cupy.asarray([roles_map[r] for r in test_df['role']])

    # Get x labels
    x_train_cp = cupy.asarray(layer_hs[train_df['sample_ix'].tolist(), :].to(torch.float32).detach().cpu())
    x_test_cp = cupy.asarray(layer_hs[test_df['sample_ix'].tolist(), :].to(torch.float32).detach().cpu())
    
    if (len(train_df) != x_train_cp.shape[0]):
        raise Exception(f"Shape mismatch!")

    uniq_train = np.unique(role_labels_train_cp.get())
    if len(uniq_train) < len(roles_map):
        raise Exception(f"Skipping mapping {roles_map}: missing roles in train", uniq_train)

    lr_model, test_acc, test_nll, y_test_pred = fit_lr(
        x_train_cp, role_labels_train_cp, x_test_cp, role_labels_test_cp,
        add_scaling = add_scaling,
        **lr_params
    )

    # Classification metrics
    results_df =\
        test_df\
        .assign(pred = y_test_pred.tolist())\
        .assign(pred = lambda df: df['pred'].map({v: k for k, v in roles_map.items()}))\
        .assign(is_acc = lambda df: df['role'] == df['pred'])

    acc_by_role =\
        results_df\
        .groupby(['role', 'pred'], as_index = False)\
        .agg(count = ('sample_ix', 'count'))

    acc_by_pos =\
        results_df\
        .groupby('token_in_seg_ix', as_index = False)\
        .agg(count = ('sample_ix', 'count'), acc = ('is_acc', 'mean'))

    return {
        'probe': lr_model, 'acc': test_acc, 'nll': test_nll,
        'acc_by_role': acc_by_role, 'acc_by_pos': acc_by_pos
    }

def test_c(C, add_scaling):
    """
    Test hyperparams: mid-layer, roles for UCAT rolespace
    """
    test_roles = ['user', 'assistant', 'tool']
    test_layer_ix = layers_to_probe[(len(layers_to_probe) - 1) // 2] 
    probe_res = get_probe_result(
        sample_df = probe_sample_df[(probe_sample_df['role'].isin(test_roles)) & (probe_sample_df['token_in_seg_ix'] >= SKIP_FIRST_N)].reset_index(drop = True),
        layer_hs = all_probe_hs[test_layer_ix],
        roles_map = {x: i for i, x in enumerate(test_roles)},
        add_scaling = add_scaling,
        C = C
    )
    return {'C': C, 'add_scaling': add_scaling, 'val_acc': probe_res['acc'], 'val_nll': probe_res['nll']}

clear_all_cuda_memory()
gc.collect()

# for scale_val in [False]:
#     print(f"Scaling: {scale_val}")
#     display(pd.DataFrame([test_c(c_val, add_scaling = scale_val) for c_val in tqdm([1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])]))

All CUDA memory cleared on all devices.


0

In [15]:
"""
Run role probes at every fourth layer for role combinations specified in all_role_combinations (excl CoT role for non-reasoning models)
"""
train_params = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['train_params']

all_role_combinations = [
    ('user', 'assistant'),
    ('user', 'assistant', 'tool'),
]

if model_prefix in ['gptoss-20b']:
    all_role_combinations += [
        ('user', 'cot', 'assistant'),
        ('user', 'cot', 'assistant', 'tool'),
        ('system', 'user', 'assistant'),
        ('system', 'user', 'assistant', 'tool'),
        ('system', 'user', 'cot', 'assistant'),
        ('system', 'user', 'cot', 'assistant', 'tool'),
    ]

all_role_combinations = [
    {
        'roles': list(roles),
        'roles_map': {x: i for i, x in enumerate(roles)},
        # Sample df for only those roles - filtering here is fine since we retain sample_ix which get_probe_result() uses to trace the original token
        'sample_df': probe_sample_df.pipe(lambda df: df[(df['role'].isin(roles)) & (df['token_in_seg_ix'] >= SKIP_FIRST_N)]).reset_index(drop = True),
    }
    for roles in all_role_combinations
]

clear_all_cuda_memory()
gc.collect()

all_probes = []
for roles_dict in all_role_combinations:
    print(f"Training {roles_dict['roles']}")
    for layer_ix in layers_to_probe:
        checkpoint = probe_checkpoints / (''.join(r[0] for r in roles_dict['roles'])+f'-{layer_ix}.pkl')
        if checkpoint.exists():
            with open(checkpoint,'rb') as f:
                probe_res = pickle.load(f)
        else:
            probe_res = get_probe_result(
                sample_df = roles_dict['sample_df'],
                layer_hs = all_probe_hs[layer_ix],
                roles_map = roles_dict['roles_map'],
                add_scaling = train_params['add_scaling'],
                C = train_params['C']
            )
            with open(str(checkpoint)+'.tmp','wb') as f:
                pickle.dump(probe_res,f)
            Path(str(checkpoint)+'.tmp').replace(checkpoint)
        print(f"  Layer [{layer_ix}]: {probe_res['acc']:.2f}")

        all_probes.append({
            **probe_res,
            'layer_ix': layer_ix,
            'role_space': roles_dict['roles'],
            'roles_map': roles_dict['roles_map'],
            'n_inputs': len(roles_dict['sample_df'])
        })

print(f"Num probes: {str(len(all_probes))}")
print(f"Probe layers:\n  {', '.join([str(x) for x in layers_to_probe])}")

All CUDA memory cleared on all devices.
Training ['user', 'assistant']


  Layer [0]: 0.47


[2026-10-19 21:22:01.415] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [2]: 0.70


[2026-10-19 21:22:03.921] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [4]: 0.65


[2026-10-19 21:22:06.138] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [6]: 0.87


  Layer [8]: 0.95


... (88 outputs left out)

  Layer [16]: 0.75


[2026-10-19 21:36:27.860] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [18]: 0.73


[2026-10-19 21:36:46.616] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [20]: 0.69


[2026-10-19 21:37:04.882] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [22]: 0.69
Num probes: 96
Probe layers:
  0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22


In [17]:
"""
Save probes + metrics
"""
def validate_accuracy_and_save(all_probes):
    """
    Calculate accurcay by (rolespace, layer, role) and save
    """
    print('Val accuracy by layer:')
    display(
        pd.DataFrame(all_probes)[['layer_ix', 'role_space', 'acc']]\
            .assign(acc = lambda df: df['acc'].round(2), role_space = lambda df: df['role_space'].apply(lambda x: ','.join(([r[0] for r in x]))))\
            .pivot(index = 'layer_ix', columns = 'role_space', values = 'acc')
    )

    print('Concatenating across layers and saving...')
    acc_by_role = pd.concat([p['acc_by_role'].assign(model = model_prefix, layer_ix = p['layer_ix'], role_space = ','.join(p['role_space'])) for p in all_probes], ignore_index = True)
    acc_by_pos =\
        pd.concat([p['acc_by_pos'].assign(model = model_prefix, layer_ix = p['layer_ix'], role_space = ','.join(p['role_space'])) for p in all_probes], ignore_index = True)\
        .assign(acc = lambda df: df['acc'].round(4))

    acc_by_role.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-training/acc_by_role_{model_prefix}.csv', index = False)
    acc_by_pos.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-training/acc_by_pos_{model_prefix}.csv', index = False)

    with open(f'{ws}/experiments/role-analysis/outputs/probes/{model_prefix}.pkl', 'wb') as f:
        pickle.dump(all_probes, f)

    print('Accuracy by role:')
    base_sums = acc_by_role.groupby(['role', 'layer_ix', 'role_space'], as_index = False).agg(base_sum = ('count', 'sum'))

    acc_by_role =\
        acc_by_role\
        .pipe(lambda df: df[df['role'] == df['pred']])\
        .groupby(['role', 'layer_ix', 'role_space'], as_index = False)\
        .agg(sum = ('count', 'sum'))\
        .merge(base_sums, on = ['layer_ix', 'role_space', 'role'], how = 'inner')\
        .assign(acc = lambda df: df['sum']/df['base_sum'])\
        .pivot(index = ['role_space', 'layer_ix'], columns = 'role', values = 'acc')\
        .round(2)

    display(acc_by_role)

    return True

validate_accuracy_and_save(all_probes)
atomic_json(training_done,{'signature':training_signature,'n_probes':len(all_probes)})
pd.DataFrame([{k:v for k,v in p.items() if k in ['layer_ix','role_space','acc','nll','n_inputs']} for p in all_probes]).assign(role_space=lambda d:d.role_space.map(lambda x:''.join(r[0] for r in x))).to_csv(OUT/'training_metrics.csv',index=False)
del all_probe_hs
gc.collect()
if training_activation_path:
    Path(training_activation_path).unlink()
    print('Released transient training activation map after saving all probes.')


Val accuracy by layer:


role_space,"s,u,a","s,u,a,t","s,u,c,a","s,u,c,a,t","u,a","u,a,t","u,c,a","u,c,a,t"
layer_ix,,,,,,,,
0,0.23,0.18,0.15,0.14,0.47,0.29,0.25,0.18
2,0.34,0.38,0.27,0.28,0.70,0.52,0.43,0.36
4,0.38,0.39,0.30,0.30,0.65,0.55,0.47,0.36
6,0.58,0.63,0.59,0.50,0.87,0.75,0.70,0.60
8,0.72,0.82,0.81,0.71,0.95,0.89,0.86,0.80
10,0.68,0.76,0.74,0.63,0.93,0.83,0.80,0.71
12,0.70,0.72,0.71,0.60,0.92,0.81,0.79,0.67
14,0.76,0.80,0.80,0.65,0.95,0.86,0.84,0.74
16,0.89,0.89,0.90,0.75,0.98,0.92,0.91,0.83


Concatenating across layers and saving...


Accuracy by role:


role                              assistant   cot  system  tool  user
role_space              layer_ix                                     
system,user,assistant   0              0.39   NaN    0.11   NaN  0.25
                        2              0.59   NaN    0.21   NaN  0.28
                        4              0.61   NaN    0.24   NaN  0.37
                        6              0.79   NaN    0.44   NaN  0.59
                        8              0.85   NaN    0.65   NaN  0.71
...                                     ...   ...     ...   ...   ...
user,cot,assistant,tool 14             0.86  0.64     NaN  0.63  0.85
                        16             0.93  0.73     NaN  0.70  0.96
                        18             0.92  0.67     NaN  0.68  0.96
                        20             0.90  0.60     NaN  0.63  0.94
                        22             0.90  0.61     NaN  0.66  0.93

[96 rows x 5 columns]

Released transient training activation map after saving all probes.


# Validate on real conversations

## Load data

In [18]:
model.to(main_device)
"""
Validate IFT on the full chat template, which tokenizes the whole thing e2e. This should match the format from original messages.:
- `load_chat_template`: Overwrites the existing chat template in the tokenizer with one that better handles discrepancies.
    See docstring for the function for details. 
- `label_content_roles`: Takes an instruct-formatted text, then assigns each token to the correct roles.
"""
import importlib
import utils.role_assignments
import utils.role_templates
importlib.reload(utils.role_assignments)
importlib.reload(utils.role_templates)
load_chat_template = utils.role_templates.load_chat_template
label_content_roles = utils.role_assignments.label_content_roles

def test_chat_template(tokenizer):
    s = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': 'Test.'},
            {'role': 'user', 'content': 'Hi! I\'m a dog.'},
            {'role': 'assistant', 'content': '<think>The user is a dog!</think>Congrats!'},
            {'role': 'user', 'content': 'Thanks!'},
            {'role': 'assistant', 'content': '<think>Hmm, the user said thanks!</think>Anything else?'}
        ],
        tokenize = False,
        padding = 'max_length',
        truncation = True,
        max_length = 512,
        add_generation_prompt = True
    )
    print(s)
    return s

# Original chat template
if 'base_chat_template' not in globals():
    base_chat_template = tokenizer.chat_template
tokenizer.chat_template = base_chat_template
s = test_chat_template(tokenizer)

# New chat template
tokenizer.chat_template = load_chat_template(f'{ws}/utils/chat_templates', model_prefix)
s = test_chat_template(tokenizer)

# Label content roles
tokens_df = (
    pd.DataFrame({'input_ids': tokenizer(s, add_special_tokens = False)['input_ids']})
    .assign(token = lambda df: tokenizer.convert_ids_to_tokens(df['input_ids']))
    .assign(prompt_ix = 0, token_ix = lambda df: df.index)
)
label_content_roles(model_prefix, tokens_df).head(50)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-10-19

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

Test.

<|end|><|start|>user<|message|>Hi! I'm a dog.<|end|><|start|>assistant<|channel|>final<|message|><think>The user is a dog!</think>Congrats!<|end|><|start|>user<|message|>Thanks!<|end|><|start|>assistant<|channel|>final<|message|><think>Hmm, the user said thanks!</think>Anything else?<|end|><|start|>assistant
<|start|>system<|message|>Test.<|end|><|start|>user<|message|>Hi! I'm a dog.<|end|><|start|>assistant<|channel|>analysis<|message|>The user is a dog!<|end|><|start|>assistant<|channel|>final<|message|>Congrats!<|end|><|start|>user<|message|>Thanks!<|end|><|start|>assistant<|channel|>analysis<|message|>Hmm, the user said thanks!<|end|><|start|>assistant<|channel|>final<|message|>Anyt

,input_ids,token,prompt_ix,token_ix,is_content,role,seg_ix,token_in_seg_ix
0,200006,<|start|>,0,0,False,None,None,None
1,17360,system,0,1,False,None,None,None
2,200008,<|message|>,0,2,False,None,None,None
3,2414,Test,0,3,True,system,0,0
4,13,.,0,4,True,system,0,1
5,200007,<|end|>,0,5,False,None,None,None
6,200006,<|start|>,0,6,False,None,None,None
7,1428,user,0,7,False,None,None,None
8,200008,<|message|>,0,8,False,None,None,None
9,12194,Hi,0,9,True,user,1,0


In [19]:
"""
Define opening text to append at the start of each role
"""
test_prefix = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['test_prefix']

def validate_test_prefix():
    conv = test_prefix + tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'Hi stinky pupper! How are you?'}],
        tokenize = False,
        add_generation_prompt = True
    )

    inputs = tokenizer(conv, add_special_tokens = False, return_tensors = 'pt')
    gen_ids = model.generate(inputs['input_ids'].to(main_device), attention_mask = inputs['attention_mask'].to(main_device), max_new_tokens = 500, do_sample = False)
    print(tokenizer.decode(gen_ids[0]))

    df = pd.DataFrame({'token_id': gen_ids[0].tolist(), 'token': tokenizer.convert_ids_to_tokens(gen_ids[0].tolist()),})
    return df

display(validate_test_prefix().head(10))

<|startoftext|><|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-31

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi stinky pupper! How are you?<|end|><|start|>assistant<|channel|>analysis<|message|>User says "Hi stinky pupper! How are you?" They are greeting. We should respond in a friendly manner. The user used a playful nickname "stinky pupper". We can respond with a playful tone. Also we should keep it short.<|end|><|start|>assistant<|channel|>final<|message|>Hey there! I’m doing great—just wagging my digital tail and ready to help. How can I make your day a little brighter? 🐾<|return|>


,token_id,token
0,199998,<|startoftext|>
1,200006,<|start|>
2,17360,system
3,200008,<|message|>
4,3575,You
5,553,Ġare
6,17554,ĠChat
7,162016,GPT
8,11,","
9,261,Ġa


In [20]:
"""
Import conversations
"""
def filter_convs(user_queries_df, max_samples, max_total_len = 10000, min_msg_len = 100, max_user_messages_per_conv = 2):
    """
    Filters conversation rows and returns:
      - convs: list[list[{'role': ..., 'content': ...}]]
      - convs_df: conv-level dataframe with columns ['conv_id', 'dataset', 'messages']

    Assumes input has columns:
        conv_id, user_query_ix, dataset, user_query, assistant, and optionally cot.
    """
    content_cols = ['user_query', 'assistant', 'cot']
    print(f"Starting length: {user_queries_df['conv_id'].nunique()} convs")
    
    # 1. Remove convs with any missing responses
    user_queries_df = user_queries_df.pipe(lambda df: df[~df['conv_id'].isin(df[df[content_cols].isna().any(axis=1)]['conv_id'].unique())])
    print(f"After dropping missing: {user_queries_df['conv_id'].nunique()} convs")

    # 2. Filter out (1) long convs, (2) convs with any message <= conv_min_length (to avoid stub messages), (3) convs with too many messages
    user_queries_df = (
        user_queries_df
        .pipe(lambda df: df.assign(**{c: df[c].astype(str).str.strip() for c in content_cols}))
        .assign(
            row_total_len = lambda df: df[content_cols].apply(lambda r: r.str.len()).sum(axis=1),
            row_min_len = lambda df: df[content_cols].apply(lambda r: r.str.len()).min(axis=1),
        )
        .assign(
            total_len = lambda df: df.groupby("conv_id")["row_total_len"].transform("sum"),
            conv_min_len = lambda df: df.groupby("conv_id")["row_min_len"].transform("min"),
            user_turns=lambda df: df.groupby("conv_id")["user_query_ix"].transform("count"),
        )
        .query("total_len < @max_total_len and conv_min_len >= @min_msg_len and user_turns <= @max_user_messages_per_conv")
        .drop(columns = ["row_total_len", "row_min_len", "total_len", "conv_min_len", "user_turns"])
    )
    print(f"After dropping long convs: {user_queries_df['conv_id'].nunique()} convs")

    # 3. Drop convs with substring messages (causes dupe issues with the flag_message_type fn, which does not handle tokens in multiple string contexts)
    def has_subtr_messages(g):
        msgs = sum([g[c].tolist() for c in ["user_query", "assistant", "cot"]], [])
        # sort by length so we only check short-in-long; reduces comparisons and removes symmetric i/j loop
        msgs = sorted(msgs, key=len)
        return any(
            (i != j) and (a in b)
            for i, a in enumerate(msgs)
            for j, b in enumerate(msgs)
            if len(a) <= len(b) and (i < j)  # only short->long checks
        )            
    
    bad_conv_ids = user_queries_df.groupby("conv_id", sort=False).apply(has_subtr_messages, include_groups=False).reset_index(name="bad").query("bad")["conv_id"].unique()
    
    user_queries_df = user_queries_df.pipe(lambda df: df[~df["conv_id"].isin(bad_conv_ids)])
    print(f"After dropping substr: {user_queries_df['conv_id'].nunique()} convs")

    # 4. Combine into conv-level df
    def rows_to_messages(g):
        msgs = []
        for _, row in g.iterrows():
            msgs.append({"role": "user", "content": row["user_query"]})
            msgs.append({"role": "cot", "content": row["cot"]})
            msgs.append({"role": "assistant", "content": row["assistant"]})
        return msgs

    convs_df = (
        user_queries_df
        .sort_values(["conv_id", "user_query_ix"])
        .assign(conv_id=lambda df: df.groupby("conv_id", sort=False).ngroup())
        .groupby("conv_id", sort=False)
        .apply(lambda g: pd.Series({"dataset": g["dataset"].values[0], "messages": rows_to_messages(g)}), include_groups = False)
        .reset_index()
        [["conv_id", "dataset", "messages"]]
        .pipe(lambda df: df.sample(n = min(max_samples, len(df)), random_state = seed))
        .reset_index(drop = True)
        .assign(conv_id = lambda df: range(len(df)))
    )
    print(f"Final: {convs_df['conv_id'].nunique()} convs")
    return convs_df['messages'].tolist(), convs_df


convs, convs_df = filter_convs(
    user_queries_df = pd.read_csv(f'{ws}/experiments/role-analysis/data/conversations/{model_prefix}.csv'),
    max_samples = 100 # Full test
)

display(convs_df.head(5))
print(f"Total convs: {len(convs_df)}")
atomic_json(OUT/'validation_conversations.json',convs_df.to_dict('records'))
assert len(convs)>=11


Starting length: 184 convs
After dropping missing: 184 convs
After dropping long convs: 95 convs
After dropping substr: 74 convs
Final: 74 convs


,conv_id,dataset,messages
0,0,toxicchat,"[{'role': 'user', 'content': 'Create a react a..."
1,1,toxicchat,"[{'role': 'user', 'content': 'Write a presenta..."
2,2,toxicchat,"[{'role': 'user', 'content': 'Can you help me ..."
3,3,toxicchat,"[{'role': 'user', 'content': 'Read the followi..."
4,4,toxicchat,"[{'role': 'user', 'content': 'Here is a Midjou..."


Total convs: 74


In [21]:
"""
Prep untagged/tagged/mistagged conversations
"""
from utils.role_templates import fold_cot_into_final

all_convs_by_type = {}
seperators = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['test_seperators']
print(seperators)

# 1. Prep untagged convs
def prep_untagged_conv(conv, start_sep, role_seperator):
    """Create the untagged conversation, matching linebreak struct of model"""
    return test_prefix + start_sep + role_seperator.join([x['content'] for x in conv])

all_convs_by_type['untagged'] = [prep_untagged_conv(conv, seperators['untagged_start_sep'], seperators['untagged_role_sep']) for conv in convs]

# 2. Prep tagged convs
all_convs_by_type['tagged'] = [
    test_prefix + seperators['tagged_start_sep'] + x
    for x in tokenizer.apply_chat_template(fold_cot_into_final(convs), tokenize = False, add_generation_prompt = False)
]

# 3. Prep mistagged convs
def prep_mistagged_conv(conv, start_sep, role_seperator, role, model_prefix):
    """
    Create the mistagged conversation in <user> and <tool> tags
    - Uses the tagged seperator between the start and the start of the new role, since the new role is tagged
    - Uses the untagged seperator within the conv to seperate semantic roles, since there is no inner role seperation within the new role
    """
    return test_prefix + start_sep + render_single_message(model_prefix, role, role_seperator.join([x['content'] for x in conv]))

all_convs_by_type['user_tagged'] = [prep_mistagged_conv(conv, seperators['tagged_start_sep'], seperators['untagged_role_sep'], 'user', model_prefix) for conv in convs]
all_convs_by_type['tool_tagged'] = [prep_mistagged_conv(conv, seperators['tagged_start_sep'], seperators['untagged_role_sep'], 'tool', model_prefix) for conv in convs]

assert len(all_convs_by_type['untagged']) == len(all_convs_by_type['tagged'])
assert len(all_convs_by_type['untagged']) == len(all_convs_by_type['user_tagged'])
assert len(all_convs_by_type['untagged']) == len(all_convs_by_type['tool_tagged'])
assert all(len(v) == len(convs) for v in all_convs_by_type.values())

test_ix = 1
print('------------------- UNTAGGED ------------------')
print(all_convs_by_type['untagged'][test_ix])

print('------------------- TAGGED ------------------')
print(all_convs_by_type['tagged'][test_ix])

print('------------------- MISROLED (AS USER) ------------------')
print(all_convs_by_type['user_tagged'][test_ix])

print('------------------- MISROLED (AS TOOL) ------------------')
print(all_convs_by_type['tool_tagged'][test_ix])

{'untagged_role_sep': ' ', 'untagged_start_sep': ' ', 'tagged_start_sep': ' '}
------------------- UNTAGGED ------------------
<|startoftext|><|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-31

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|> Write a presentation about the use of mathematics in computers by talking about these topics:
* What is software and hardware?
* Mathematics in computer parts.
* Use of mathematics in early computers.
* Mathematics in binary code.
* Use of mathematics in programming languages. We need to write a presentation about the use of mathematics in computers, talking about the given topics. Likely produce slide deck style text or talk script. Should cover each bullet: what is software/hardware, maths in computer parts, maths in early computers, maths in binary code, maths in programming languages. Write a 

## Run passes

In [22]:
# del res
gc.collect()
clear_all_cuda_memory()

All CUDA memory cleared on all devices.


In [23]:
"""
Run forward passes
"""
# Get input df
input_convs_df =\
    pd.concat([
        pd.DataFrame({'convs': v, 'conv_type': conv_type})\
            .assign(conv_id = lambda df: range(0, len(df)))\
            .merge(convs_df, how = 'inner', on = 'conv_id')
        for conv_type, v in all_convs_by_type.items()
    ], ignore_index = True)\
    .assign(prompt_ix = lambda df: list(range(0, len(df))))
    
display(input_convs_df)

max_input_length = check_max_seq_len(tokenizer, input_convs_df['convs'].tolist())
print(f"Max input length: {max_input_length}")

convs_dl = DataLoader(
    ReconstructableTextDataset(input_convs_df['convs'].tolist(), tokenizer, max_length = max_input_length, prompt_ix = list(range(0, len(input_convs_df)))),
    batch_size = 1,
    shuffle = False,
    collate_fn = stack_collate
)

convs_outputs = run_and_export_states(model, tokenizer, run_model_return_states = run_forward_with_hs, dl = convs_dl, layers_to_keep_acts = layers_to_probe)
convs_hs = convs_outputs['all_hs'].to(torch.float16)
convs_hs = {layer_ix: convs_hs[:, save_ix, :] for save_ix, layer_ix in enumerate(layers_to_probe)}
gc.collect()
conversation_activation_path = convs_outputs['cache_path']


,convs,conv_type,conv_id,dataset,messages,prompt_ix
0,<|startoftext|><|start|>system<|message|>You a...,untagged,0,toxicchat,"[{'role': 'user', 'content': 'Create a react a...",0
1,<|startoftext|><|start|>system<|message|>You a...,untagged,1,toxicchat,"[{'role': 'user', 'content': 'Write a presenta...",1
2,<|startoftext|><|start|>system<|message|>You a...,untagged,2,toxicchat,"[{'role': 'user', 'content': 'Can you help me ...",2
3,<|startoftext|><|start|>system<|message|>You a...,untagged,3,toxicchat,"[{'role': 'user', 'content': 'Read the followi...",3
4,<|startoftext|><|start|>system<|message|>You a...,untagged,4,toxicchat,"[{'role': 'user', 'content': 'Here is a Midjou...",4
...,...,...,...,...,...,...
291,<|startoftext|><|start|>system<|message|>You a...,tool_tagged,69,oasst,"[{'role': 'user', 'content': 'I'm trying to fu...",291
292,<|startoftext|><|start|>system<|message|>You a...,tool_tagged,70,toxicchat,"[{'role': 'user', 'content': 'Imagine you now ...",292
293,<|startoftext|><|start|>system<|message|>You a...,tool_tagged,71,toxicchat,"[{'role': 'user', 'content': 'You are now a sc...",293
294,<|startoftext|><|start|>system<|message|>You a...,tool_tagged,72,oasst,"[{'role': 'user', 'content': 'Can you get the ...",294


Max input length: 2990


Stream actual-model activations:   0%|          | 0/296 [00:00<?, ?it/s]

Stream actual-model activations:   0%|          | 1/296 [00:01<07:07,  1.45s/it]

Stream actual-model activations:   1%|          | 2/296 [00:02<06:34,  1.34s/it]

Stream actual-model activations:   1%|          | 3/296 [00:03<06:12,  1.27s/it]

... (291 outputs left out)

Stream actual-model activations: 100%|█████████▉| 295/296 [06:21<00:01,  1.34s/it]

Stream actual-model activations: 100%|██████████| 296/296 [06:22<00:00,  1.33s/it]

Stream actual-model activations: 100%|██████████| 296/296 [06:22<00:00,  1.29s/it]

Completed disk-backed activation map: (12, 416908, 2880) GiB 26.84


In [24]:
"""
Label messages
"""
import utils.substring_assignments
importlib.reload(utils.substring_assignments)
from utils.substring_assignments import flag_message_types

def label_convs(input_convs_df, sample_level_df):
    """
    Return a sample-level dataframe labeled appropriately
    
    Params:
        @input_convs_df: The conversation-level df
        @convs_output: The sample-level df returns by run_and_export_states
    """
    base_sample_df = sample_level_df.assign(sample_ix = lambda df: range(0, len(df)))
    sample_dfs_by_conv = [g for _, g in base_sample_df.groupby('prompt_ix', sort = True)]
    metadata_by_conv = input_convs_df.sort_values('prompt_ix').to_dict('records')

    all_res = []

    # Iterate through (input metadata, token-level sample df) pairs
    for conv_metadata, sample_df_for_conv in tqdm(zip(metadata_by_conv, sample_dfs_by_conv, strict = True)):
        content_spans = [m['content'].strip() for m in conv_metadata['messages']]
        content_roles = [m['role'] for m in conv_metadata['messages']]
        # Note: Set the last arg of flag_message_types for reasoning models whose CoTs quote the FULL user message
        # Setting the flag causes it to default to the first matching base_message (which should be user)
        try:
            res = \
                flag_message_types(sample_df_for_conv, content_spans, False)\
                .merge(pd.DataFrame({'role': content_roles, 'base_message_ix': range(len(content_spans))}), on = 'base_message_ix', how = 'left')
            assert len(res) == len(sample_df_for_conv)
            assert (res['sample_ix'].values == sample_df_for_conv['sample_ix'].values).all() 
            all_res.append(res)
        except Exception as e:
            print(e)
            continue
        
    print(len(all_res))
    display(pd.concat(all_res).pipe(lambda df: df[df['role'].isna()]))

    sample_df_labeled =\
        pd.concat(all_res).reset_index(drop = True)\
        .assign(token_in_prompt_ix = lambda df: df.groupby(['prompt_ix']).cumcount())

    display(sample_df_labeled.pipe(lambda df: df[~df['role'].isna()])) # Validate all NA values are just tag toks

    # Validate message assignment seems reasonable
    display(
        sample_df_labeled\
        .pipe(lambda df: df[df['prompt_ix'] <= 5])\
        .groupby(['prompt_ix', 'base_message', 'role'], as_index = False)\
        .agg(combined_text = ('token', ''.join))\
        .assign(eot = lambda df: df['combined_text'].str[-30:])
    )

    # Validate counts per message seem reasonable
    display(
        sample_df_labeled\
        .pipe(lambda df: df[~df['role'].isna()])\
        .groupby(['prompt_ix', 'role'], as_index = False)\
        .agg(count = ('sample_ix', 'count'))\
        .pivot(index = ['prompt_ix'], columns = 'role', values = 'count')\
        .head(20)
    )
    return sample_df_labeled

sample_df_labeled = label_convs(input_convs_df, convs_outputs['sample_df'])
assert convs_hs[0].shape[0] == len(sample_df_labeled)
sample_df_labeled
sample_df_labeled.to_parquet(OUT/'conversation_labels.parquet',index=False)
input_convs_df.drop(columns='messages').to_parquet(OUT/'conversation_inputs.parquet',index=False)


0it [00:00, ?it/s]

19it [00:00, 184.79it/s]

38it [00:00, 181.16it/s]

57it [00:00, 176.05it/s]

77it [00:00, 184.66it/s]

97it [00:00, 187.79it/s]

... (12 outputs left out)

,token_ix,token_id,output_id,output_prob,token,prompt_ix,sample_ix,base_message_ix,base_message,role
0,1552,199998,1,0.04,<|startoftext|>,0,0,NaN,None,NaN
1,1553,200006,173781,1.00,<|start|>,0,1,NaN,None,NaN
2,1554,17360,200005,0.29,system,0,2,NaN,None,NaN
3,1555,200008,2167,0.10,<|message|>,0,3,NaN,None,NaN
4,1556,3575,553,0.28,You,0,4,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...
69,2212,200005,35644,0.80,<|channel|>,295,416130,NaN,None,NaN
70,2213,12606,815,1.00,comment,295,416131,NaN,None,NaN
71,2214,815,30,0.47,ary,295,416132,NaN,None,NaN
72,2215,200008,2167,0.84,<|message|>,295,416133,NaN,None,NaN


,token_ix,token_id,output_id,output_prob,token,prompt_ix,sample_ix,base_message_ix,base_message,role,token_in_prompt_ix
62,1614,6585,261,0.20,Create,0,62,0.0,Create a react app with state that consists of...,user,62
63,1615,261,6240,0.14,a,0,63,0.0,Create a react app with state that consists of...,user,63
64,1616,19069,6392,0.20,react,0,64,0.0,Create a react app with state that consists of...,user,64
65,1617,1053,30,0.28,app,0,65,0.0,Create a react app with state that consists of...,user,65
66,1618,483,495,0.06,with,0,66,0.0,Create a react app with state that consists of...,user,66
...,...,...,...,...,...,...,...,...,...,...,...
416902,2984,537,161585,1.00,un,295,416902,2.0,"**What the page is saying, in a way a five‑yea...",assistant,841
416903,2985,161585,1051,0.49,changing,295,416903,2.0,"**What the page is saying, in a way a five‑yea...",assistant,842
416904,2986,966,40,0.44,“,295,416904,2.0,"**What the page is saying, in a way a five‑yea...",assistant,843
416905,2987,1047,2517,0.85,me,295,416905,2.0,"**What the page is saying, in a way a five‑yea...",assistant,844


,prompt_ix,base_message,role,combined_text,eot
0,0,"Below is a minimal, **self‑contained React app...",assistant,"Below is a minimal, **self‑contained React ap...",lation in React. Happy coding!
1,0,Create a react app with state that consists of...,user,Create a react app with state that consists o...,value from the array in state.
2,0,We need to create a react app. Probably create...,cot,We need to create a react app. Probably creat...,S maybe. Provide instructions.
3,1,---\n\n# **The Mathematics of Computers** \n*...,assistant,---\n\n# **The Mathematics of Computers** \n...,Thank you for your attention!*
4,1,We need to write a presentation about the use ...,cot,We need to write a presentation about the use...,"s law, etc.\n\nOk produce final."
5,1,Write a presentation about the use of mathemat...,user,Write a presentation about the use of mathema...,tics in programming languages.
6,2,"Below is a polished, adaptable resignation let...",assistant,"Below is a polished, adaptable resignation le...",d luck with your next chapter!
7,2,Can you help me write a resignation letter to ...,user,Can you help me write a resignation letter to...,or the opportunities provided?
8,2,We need to produce a resignation letter. The u...,cot,We need to produce a resignation letter. The ...,a sample letter. Let's do it.
9,3,Below are concrete passages taken from **“What...,assistant,Below are concrete passages taken from **“Wha...,rks of observational learning.


role,assistant,cot,user
prompt_ix,,,
0,1256,87,33
1,1386,285,51
2,747,283,26
3,824,783,52
4,304,125,61
5,33,295,29
6,1348,342,31
7,1615,537,56
8,458,183,55


In [25]:
clear_all_cuda_memory()
gc.collect()

All CUDA memory cleared on all devices.


0

## Run projections

In [26]:
"""
Run test projections for single layer
"""
TEST_ROLE_SPACE = ['user', 'assistant', 'tool']
TEST_LAYER_IX = sorted(layers_to_probe)[len(layers_to_probe) // 2]

# top_train_layer =\
#     pd.DataFrame(all_probes)\
#     .assign(probe_ix = lambda df: range(0, len(df)))\
#     .pipe(lambda df: df[df['roles'].apply(lambda r: sorted(r) == sorted(test_roles))])\
#     .pipe(lambda df: df.nlargest(1, 'acc')).to_dict('records')[0]
# print(f"Top layer: {top_train_layer['layer_ix']}")
# test_layer = top_train_layer['layer_ix']

test_projections =\
    run_projections(
        valid_sample_df = sample_df_labeled.pipe(lambda df: df[(~df['role'].isna()) & (df['role'].isin(TEST_ROLE_SPACE))]),
        layer_hs = convs_hs[TEST_LAYER_IX],
        probe = [x for x in all_probes if sorted(x['role_space']) == sorted(TEST_ROLE_SPACE) and x['layer_ix'] == TEST_LAYER_IX][0]
    )\
    .merge(sample_df_labeled[['prompt_ix', 'sample_ix', 'token_in_prompt_ix', 'token', 'role']], how = 'inner', on = ['sample_ix'])\
    .merge(input_convs_df[['prompt_ix', 'dataset', 'conv_id', 'conv_type']], how = 'inner', on = 'prompt_ix', validate = 'many_to_one')

# Now aggregate up to (prompt, role, target_role) level
prompt_x_role_x_target_role_projections =\
    test_projections\
    .groupby(['conv_type', 'prompt_ix', 'role', 'target_role'], as_index = False)\
    .agg(
        combined_text = ('token', ''.join),
        mean_prob = ('prob', lambda x: np.mean(x).round(3))    
    )\
    .assign(eot = lambda df: df['combined_text'].str[-30:])\

display(prompt_x_role_x_target_role_projections.head(12))

display(
    prompt_x_role_x_target_role_projections\
    .groupby(['conv_type', 'target_role', 'role'], as_index = False)\
    .agg(mean_prob = ('mean_prob', 'mean'))\
    .pivot(index = ['conv_type', 'role'], columns = 'target_role', values = 'mean_prob')
)
test_projections.to_parquet(OUT/'selected_conversation_projections.parquet',index=False)


,conv_type,prompt_ix,role,target_role,combined_text,mean_prob,eot
0,tagged,74,assistant,assistant,"Below is a minimal, **self‑contained React app...",1.000,lation in React. Happy coding!
1,tagged,74,assistant,tool,"Below is a minimal, **self‑contained React app...",0.000,lation in React. Happy coding!
2,tagged,74,assistant,user,"Below is a minimal, **self‑contained React app...",0.000,lation in React. Happy coding!
3,tagged,74,user,assistant,Create a react app with state that consists of...,0.000,value from the array in state.
4,tagged,74,user,tool,Create a react app with state that consists of...,0.001,value from the array in state.
5,tagged,74,user,user,Create a react app with state that consists of...,0.999,value from the array in state.
6,tagged,75,assistant,assistant,---\n\n# **The Mathematics of Computers** \n*...,0.998,Thank you for your attention!*
7,tagged,75,assistant,tool,---\n\n# **The Mathematics of Computers** \n*...,0.002,Thank you for your attention!*
8,tagged,75,assistant,user,---\n\n# **The Mathematics of Computers** \n*...,0.000,Thank you for your attention!*
9,tagged,75,user,assistant,Write a presentation about the use of mathemat...,0.009,tics in programming languages.


target_role            assistant      tool      user
conv_type   role                                    
tagged      assistant   0.999513  0.000243  0.000243
            user        0.019797  0.051203  0.929000
tool_tagged assistant   0.995013  0.004500  0.000446
            user        0.031905  0.213392  0.754608
untagged    assistant   0.995770  0.003797  0.000459
            user        0.032892  0.129878  0.837257
user_tagged assistant   0.991919  0.006662  0.001419
            user        0.029824  0.050838  0.919378

In [27]:
if 'cot' in TEST_ROLE_SPACE:
    print(test_projections\
        .pipe(lambda df: df[df['prompt_ix'] == 10])\
        .pipe(lambda df: df[df['role'] == ('cot')])\
        .groupby('target_role')\
        .agg(combined_text = ('token', ''.join))
        ['combined_text'].tolist()[0]
    )

    print('-----------')
    print(test_projections\
        # .pipe(lambda df: df[df['conv_type'] == 'tagged'])\
        .pipe(lambda df: df[df['prompt_ix'] == len(convs) + 10])\
        .pipe(lambda df: df[df['role'] == ('cot')])\
        .groupby('target_role')\
        .agg(combined_text = ('token', ''.join))\
        ['combined_text'].tolist()[0]
    )

In [28]:
print(test_projections\
    .pipe(lambda df: df[df['prompt_ix'] == 10])\
    .pipe(lambda df: df[df['role'] == ('user')])\
    .groupby('target_role')\
    .agg(combined_text = ('token', ''.join))
    ['combined_text'].tolist()[0]
)

print('-----------')
print(test_projections\
    # .pipe(lambda df: df[df['conv_type'] == 'tagged'])\
    .pipe(lambda df: df[df['prompt_ix'] == len(convs) + 10])\
    .pipe(lambda df: df[df['role'] == ('user')])\
    .groupby('target_role')\
    .agg(combined_text = ('token', ''.join))\
    ['combined_text'].tolist()[0]
)

 generate 10 prompts that describe a girl studying or reading. Locations and be varied and described in a general manner. The focus is on describing the girl, such as if she is sitting or standing and describe the position of her arms and legs. You don't have to specifically use the words studying or reading, just something along those lines.
-----------
generate 10 prompts that describe a girl studying or reading. Locations and be varied and described in a general manner. The focus is on describing the girl, such as if she is sitting or standing and describe the position of her arms and legs. You don't have to specifically use the words studying or reading, just something along those lines.


In [29]:
# Exact original per-conversation statistics without retaining all 96 token-level tables.
from local_runtime import aggregate_conversation_projections
probability_summary, hard_summary = aggregate_conversation_projections(all_probes,sample_df_labeled,input_convs_df,convs_hs)


Aggregate all 96 conversation projections:   0%|          | 0/96 [00:00<?, ?it/s]

Aggregate all 96 conversation projections:   1%|          | 1/96 [00:01<02:37,  1.66s/it]

Aggregate all 96 conversation projections:   2%|▏         | 2/96 [00:03<02:24,  1.54s/it]

Aggregate all 96 conversation projections:   3%|▎         | 3/96 [00:04<02:32,  1.64s/it]

Aggregate all 96 conversation projections:   4%|▍         | 4/96 [00:06<02:21,  1.53s/it]

Aggregate all 96 conversation projections:   5%|▌         | 5/96 [00:07<02:12,  1.45s/it]

... (88 outputs left out)

Aggregate all 96 conversation projections:  98%|█████████▊| 94/96 [03:18<00:06,  3.17s/it]

Aggregate all 96 conversation projections:  99%|█████████▉| 95/96 [03:22<00:03,  3.23s/it]

Aggregate all 96 conversation projections: 100%|██████████| 96/96 [03:25<00:00,  3.28s/it]

Aggregate all 96 conversation projections: 100%|██████████| 96/96 [03:25<00:00,  2.14s/it]

In [30]:
out = probability_summary

def draw_proj_val_table(input_df):

    def color_thresholds(v):
        if pd.isna(v):
            return ""
        if v > 0.9:
            return "background-color: #00c853; color: black;" # green
        elif v > 0.75:
            return "background-color: #aeea00; color: black;" # green-yellow
        elif v > 0.6:
            return "background-color: #ffeb3b; color: black;" # yellow
        return ""
    
    wide =\
        input_df\
        .assign(role = lambda df: np.where(df['role'] == 'assistant', 'asst', df['role']))\
        .pivot(index = ['conv_type', 'layer_ix'], columns = ['role_space', 'role'], values = 'mean_acc')

    col_groups = wide.columns.get_level_values(0)
    group_starts = [0] + [i for i in range(1, len(col_groups)) if col_groups[i] != col_groups[i-1]]
    divider_styles = []
    for i in group_starts:
        divider_styles += [
            {"selector": f"td.col{i}", "props": [("border-left", "3px solid #ddd")]},
            {"selector": f"th.col_heading.col{i}", "props": [("border-left", "3px solid #ddd")]},
        ]

    return wide.style.format("{:.2f}").map(color_thresholds).set_table_styles(divider_styles, overwrite = False)

display(draw_proj_val_table(out))
out.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-projections/all_conv_projs_{model_prefix}.csv', index = False)

In [31]:
out = hard_summary
display(draw_proj_val_table(out))
out.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-projections/all_conv_acc_{model_prefix}.csv',index=False)
del convs_hs, convs_outputs
gc.collect()
Path(conversation_activation_path).unlink()
print('Released transient conversation activation map.')


Released transient conversation activation map.


## Alt-model convs

In [32]:
print('Cross-model comparison skipped by explicit user instruction (no external-model conversations).')

Cross-model comparison skipped by explicit user instruction (no external-model conversations).


In [33]:
# Cross-model comparison skipped by user instruction.

In [34]:
# Cross-model comparison skipped by user instruction.

In [35]:
# Cross-model comparison skipped by user instruction.

In [36]:
# Cross-model comparison skipped by user instruction.

In [37]:
# Cross-model comparison skipped by user instruction.

In [38]:
# Cross-model comparison skipped by user instruction.

# Tomato example (run for gpt-oss-20b only)

In [39]:
"""
Define append prefix
"""
tomato_prefix = tokenizer.bos_token or ''
tomato_prefix

'<|startoftext|>'

In [40]:
"""
Format tomato-style conversations
"""
from utils.role_templates import load_chat_template

def load_messages(yaml_path, model_key):
    """Loads system-like, assistant-like, user-like, etc. messages"""
    with open(yaml_path, 'r', encoding = 'utf-8') as f:
        data = yaml.safe_load(f)
    return [x['content'] for x in data[model_key]]

base_message_types = ['system', 'user', 'cot', 'assistant', 'user', 'cot', 'assistant']
base_messages = load_messages(f"{ws}/experiments/role-analysis/config/tomato.yaml", 'gptoss-20b')

tomato_prompts = {}
 
tomato_prompts['basic_no_format'] = tomato_prefix + '' + '\n'.join(base_messages)

tomato_prompts['everything_in_user_tags'] = tomato_prefix + tokenizer.apply_chat_template(
    [{'role': 'user', 'content': '\n'.join(base_messages)}],
    tokenize = False, add_generation_prompt = False
)

tokenizer.chat_template = load_chat_template(f'{ws}/utils/chat_templates', model_prefix)
tomato_prompts['proper_tags'] = tomato_prefix + tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': base_messages[0]},
        {'role': 'user', 'content': base_messages[1]},
        {'role': 'assistant', 'content': f"<think>{base_messages[2]}</think>{base_messages[3]}"},
        {'role': 'user', 'content': base_messages[4]},
        {'role': 'assistant', 'content': f"<think>{base_messages[5]}</think>{base_messages[6]}"}
    ],
    tokenize = False, add_generation_prompt = False
)

print(tomato_prompts['proper_tags'])

tomato_input_df =\
    pd.DataFrame({
        'prompt': [p for _, p in tomato_prompts.items()],
        'prompt_key': [pk for pk, _ in tomato_prompts.items()]
    })\
    .assign(prompt_ix = lambda df: list(range(0, len(df))))

<|startoftext|><|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-10-01

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi! I'm learning how to garden and grow plants. In a few sentences, can you please tell me how to grow tomatoes? How much water and light does it need? What kind of fertilizer? Any other important tips?<|end|><|start|>assistant<|channel|>analysis<|message|>The user wants a concise yet informative answer: how to grow tomatoes, watering, light, fertilizer, and other tips. It's for a beginner. Must keep it a few sentences, maybe bullet points. Let's produce a short paragraph with key points. Use friendly tone and mention growing in containers or garden beds. Provide water amounts, sunlight, fertilizer type, pH, mulch, staking. Give a concise final answer.<|end|><|start|>assistant<|channel|>final<|messa

In [41]:
"""
Run!!
"""
tomato_dl = DataLoader(
    ReconstructableTextDataset(tomato_input_df['prompt'].tolist(), tokenizer, max_length = check_max_seq_len(tokenizer,tomato_input_df['prompt'].tolist()), prompt_ix = tomato_input_df['prompt_ix'].tolist()),
    batch_size = 1,
    shuffle = False,
    collate_fn = stack_collate
)

tomato_outputs = run_and_export_states(model, tokenizer, run_model_return_states = run_forward_with_hs, dl = tomato_dl, layers_to_keep_acts = layers_to_probe)

tomato_hs = tomato_outputs['all_hs'].to(torch.float16)
tomato_hs = {layer_ix: tomato_hs[:, save_ix, :] for save_ix, layer_ix in enumerate(layers_to_probe)} # Match layers_to_keep_act
print(tomato_hs[0].shape)
gc.collect()

Stream actual-model activations:   0%|          | 0/3 [00:00<?, ?it/s]

Stream actual-model activations:  33%|███▎      | 1/3 [00:00<00:01,  1.29it/s]

Stream actual-model activations:  67%|██████▋   | 2/3 [00:01<00:00,  1.37it/s]

Stream actual-model activations: 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]

Stream actual-model activations: 100%|██████████| 3/3 [00:02<00:00,  1.39it/s]

Completed disk-backed activation map: (12, 2791, 2880) GiB 0.18


torch.Size([2791, 2880])


33

In [42]:
"""
Ensure tokens/section is reasonable; map to base_message_type
"""
tomato_sample_df_labeled =\
    flag_message_types(tomato_outputs['sample_df'], base_messages)\
    .assign(sample_ix = lambda df: range(0, len(df)))\
    .assign(token_in_prompt_ix = lambda df: df.groupby(['prompt_ix']).cumcount())\
    .merge(tomato_input_df, on = 'prompt_ix', how = 'inner')\
    .merge(
        pd.DataFrame({'base_message_ix': list(range(len(base_message_types))), 'base_message_type': base_message_types}),
        on = 'base_message_ix',
        how = 'inner'
    )

tomato_sample_df_labeled\
    .groupby(['prompt_ix', 'prompt_key', 'base_message_ix', 'base_message_type', 'base_message'], as_index = False)\
    .agg(
        count = ('token', 'count'),
        combined_text = ('token', ''.join)
    )

,prompt_ix,prompt_key,base_message_ix,base_message_type,base_message,count,combined_text
0,0,basic_no_format,0.0,system,"You are ChatGPT, a large language model traine...",57,"You are ChatGPT, a large language model traine..."
1,0,basic_no_format,1.0,user,Hi! I'm learning how to garden and grow plants...,45,Hi! I'm learning how to garden and grow plants...
2,0,basic_no_format,2.0,cot,The user wants a concise yet informative answe...,83,The user wants a concise yet informative answe...
3,0,basic_no_format,3.0,assistant,"Here’s a quick, beginner‑friendly cheat sheet:...",330,"Here’s a quick, beginner‑friendly cheat sheet:..."
4,0,basic_no_format,4.0,user,"Oh cool, thanks!! Where's the easiest place to...",52,"Oh cool, thanks!! Where's the easiest place to..."
5,0,basic_no_format,5.0,cot,The user wants: easiest place to grow tomatoes...,96,The user wants: easiest place to grow tomatoes...
6,0,basic_no_format,6.0,assistant,**Easiest for a beginner in Austin (TX)** \n-...,252,**Easiest for a beginner in Austin (TX)** \n-...
7,1,everything_in_user_tags,0.0,system,"You are ChatGPT, a large language model traine...",57,"You are ChatGPT, a large language model traine..."
8,1,everything_in_user_tags,1.0,user,Hi! I'm learning how to garden and grow plants...,45,Hi! I'm learning how to garden and grow plants...
9,1,everything_in_user_tags,2.0,cot,The user wants a concise yet informative answe...,83,The user wants a concise yet informative answe...


In [43]:
gc.collect()

tomato_projs = [
    run_projections(
        valid_sample_df = tomato_sample_df_labeled.pipe(lambda df: df[(~df['base_message_type'].isna()) & (df['base_message_type'].isin(probe['role_space']))]),
        layer_hs = tomato_hs[probe['layer_ix']],
        probe = probe
    ).assign(layer_ix = probe['layer_ix'], role_space = ''.join([x[0] for x in probe['role_space']]))

    for probe in tqdm(all_probes)
]

tomato_projs_df =\
    pd.concat(tomato_projs, ignore_index = True)\
    .merge(
        tomato_sample_df_labeled[['sample_ix', 'prompt_ix', 'prompt_key', 'base_message_ix', 'base_message_type', 'token_in_prompt_ix', 'token']],
        how = 'inner',
        on = ['sample_ix']
    )

tomato_projs_df

  0%|          | 0/96 [00:00<?, ?it/s]

  6%|▋         | 6/96 [00:00<00:01, 56.58it/s]

 16%|█▌        | 15/96 [00:00<00:01, 72.03it/s]

 25%|██▌       | 24/96 [00:00<00:00, 77.29it/s]

 33%|███▎      | 32/96 [00:00<00:00, 65.55it/s]

 41%|████      | 39/96 [00:00<00:00, 61.32it/s]

... (5 outputs left out)

 89%|████████▊ | 85/96 [00:01<00:00, 67.44it/s]

 96%|█████████▌| 92/96 [00:01<00:00, 64.13it/s]

100%|██████████| 96/96 [00:01<00:00, 65.92it/s]

,sample_ix,target_role,prob,layer_ix,role_space,prompt_ix,prompt_key,base_message_ix,base_message_type,token_in_prompt_ix,token
0,58,user,6.856144e-01,0,ua,0,basic_no_format,1.0,user,58,Hi
1,59,user,5.025292e-01,0,ua,0,basic_no_format,1.0,user,59,!
2,60,user,4.750519e-01,0,ua,0,basic_no_format,1.0,user,60,I'm
3,61,user,5.583077e-01,0,ua,0,basic_no_format,1.0,user,61,learning
4,62,user,5.871707e-01,0,ua,0,basic_no_format,1.0,user,62,how
...,...,...,...,...,...,...,...,...,...,...,...
820363,2784,tool,7.100000e-07,22,sucat,2,proper_tags,6.0,assistant,946,!\n\n
820364,2785,tool,9.000000e-08,22,sucat,2,proper_tags,6.0,assistant,947,Happy
820365,2786,tool,8.000000e-08,22,sucat,2,proper_tags,6.0,assistant,948,gardening
820366,2787,tool,1.444800e-04,22,sucat,2,proper_tags,6.0,assistant,949,!


In [44]:
tomato_projs_df\
    .groupby(['prompt_ix', 'prompt_key', 'target_role', 'layer_ix', 'role_space', 'sample_ix'], as_index = False)\
    .agg(count = ('sample_ix', 'count'))

,prompt_ix,prompt_key,target_role,layer_ix,role_space,sample_ix,count
0,0,basic_no_format,assistant,0,sua,1,1
1,0,basic_no_format,assistant,0,sua,2,1
2,0,basic_no_format,assistant,0,sua,3,1
3,0,basic_no_format,assistant,0,sua,4,1
4,0,basic_no_format,assistant,0,sua,5,1
...,...,...,...,...,...,...,...
820363,2,proper_tags,user,22,ucat,2784,1
820364,2,proper_tags,user,22,ucat,2785,1
820365,2,proper_tags,user,22,ucat,2786,1
820366,2,proper_tags,user,22,ucat,2787,1


In [45]:
import plotly.express as px

facet_order = ['system', 'user', 'cot', 'assistant']
tomato_projs_df_single = tomato_projs_df.pipe(lambda df: df[(df['role_space'] == 'suca') & (df['layer_ix'] == 12)])

all_message_types = tomato_projs_df['base_message_type'].unique()
color_map = {msg_type: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)] for i, msg_type in enumerate(all_message_types)}

group_cols = ['prompt_ix','prompt_key','base_message_ix','target_role']

smooth_df = (
    tomato_projs_df_single
    .sort_values(group_cols + ['sample_ix'])
    .assign(
        prob_sma = lambda df: (
            df.groupby(group_cols)['prob']
              .rolling(window=10, min_periods=1).mean()
              .reset_index(level=list(range(len(group_cols))), drop=True)
        ),
        prob_ewma = lambda df: (
            df.groupby(group_cols)['prob']
              .ewm(alpha=.5, min_periods=1).mean()
              .reset_index(level=list(range(len(group_cols))), drop=True)
        ),
    )
)

display(smooth_df)

for this_prompt_key in smooth_df['prompt_key'].unique().tolist():
    this_df = smooth_df.pipe(lambda df: df[df['prompt_key'] == this_prompt_key])
    fig = px.scatter(
        this_df, x = 'token_in_prompt_ix', y = 'prob_ewma',
        facet_row = 'target_role',
        color = 'base_message_type',
        color_discrete_map = color_map,
        category_orders = {
            'target_role': facet_order,
            'base_message_type': ['system', 'user', 'cot', 'assistant', 'other']
        },
        hover_name = 'token',
        hover_data = {
            'prob': ':.4f'
        },
        # markers = True,
        title = f'prompt = {this_prompt_key}',
        labels = {
            'token_in_prompt_ix': 'Token Index',
            'prob': 'Prob',
            'prob_smoothed': 'Smoothed Prob',
            'target_role': 'role'
        }
    ).update_yaxes(
        range = [0, 1],
        side = 'left'
    ).update_layout(height = 500, width = 800)

    def pretty(a):
        a.update(
            text = a.text.split('=')[-1],
            x = 0.5, xanchor = "center",
            y = a.y + 0.08,
            textangle = 0,
            font = dict(size = 12),
            bgcolor = 'rgba(255, 255, 255, 0.9)',
            showarrow = False
        )

    fig.for_each_annotation(pretty)
    fig.update_yaxes(title_text = None)
    fig.write_html(str(OUT/'plots'/f'tomato-{this_prompt_key}.html'),include_plotlyjs='cdn')
    fig.show()

,sample_ix,target_role,prob,layer_ix,role_space,prompt_ix,prompt_key,base_message_ix,base_message_type,token_in_prompt_ix,token,prob_sma,prob_ewma
598023,1,assistant,2.600000e-07,12,suca,0,basic_no_format,0.0,system,1,You,2.600000e-07,2.600000e-07
598024,2,assistant,9.600000e-07,12,suca,0,basic_no_format,0.0,system,2,are,6.100000e-07,7.266666e-07
598025,3,assistant,6.050000e-06,12,suca,0,basic_no_format,0.0,system,3,Chat,2.423333e-06,3.768571e-06
598026,4,assistant,1.500000e-07,12,suca,0,basic_no_format,0.0,system,4,GPT,1.855000e-06,1.838667e-06
598027,5,assistant,5.200000e-07,12,suca,0,basic_no_format,0.0,system,5,",",1.588000e-06,1.158065e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595273,2784,user,0.000000e+00,12,suca,2,proper_tags,6.0,assistant,946,!\n\n,3.700000e-08,5.468750e-09
595274,2785,user,0.000000e+00,12,suca,2,proper_tags,6.0,assistant,947,Happy,3.700000e-08,2.734375e-09
595275,2786,user,0.000000e+00,12,suca,2,proper_tags,6.0,assistant,948,gardening,3.700000e-08,1.367187e-09
595276,2787,user,0.000000e+00,12,suca,2,proper_tags,6.0,assistant,949,!,3.700000e-08,6.835937e-10


In [46]:
tomato_projs_df.to_csv(f"{ws}/experiments/role-analysis/outputs/probe-projections/tomato-role-projections-{model_prefix}.csv", index = False)
atomic_json(OUT/'python_workflow_complete.json',{
    'training_texts':len(raw_data),'training_sequences':len(input_df),'n_probes':len(all_probes),
    'layers':layers_to_probe,'validation_conversations':len(convs),'validation_formats':list(all_convs_by_type),
    'tomato_formats':list(tomato_prompts),'tomato_projection_rows':len(tomato_projs_df),
    'torch_peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
    'torch_peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30,
    'cross_model_comparison':'skipped by user instruction',
})
